In [0]:
# CELL 1: Setup, Parameters, & Append-Only Audit Logger
import re
import json
import time
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType, DoubleType, BooleanType
)
from sodapy import Socrata

dbutils.widgets.text("socrata_domain", "opendata.maryland.gov", "Socrata Domain")
dbutils.widgets.text("dataset_id", "", "Single Dataset ID")
dbutils.widgets.text("catalog_name", "open_data_portal", "Catalog Name")
dbutils.widgets.text("schema_name", "bronze", "Data Schema Name")
dbutils.widgets.text("meta_schema", "meta", "Control Schema Name")
dbutils.widgets.dropdown("verbose", "true", ["true", "false"], "Verbose Logging")

SOCRATA_DOMAIN = dbutils.widgets.get("socrata_domain")
DATASET_ID = dbutils.widgets.get("dataset_id").strip()
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
META_SCHEMA = dbutils.widgets.get("meta_schema")
VERBOSE = dbutils.widgets.get("verbose").lower() == "true"

QUEUE_TABLE = f"`{CATALOG_NAME}`.`{META_SCHEMA}`.`_socrata_ingestion_queue`"

if not DATASET_ID:
    raise ValueError("dataset_id widget must be provided for single-task ingestion.")

def log(*args, **kwargs):
    if VERBOSE:
        print(*args, **kwargs)

def sanitize_table_name(name: str) -> str:
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', (name or '').lower())
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean[:128]

def sql_str(val) -> str:
    if val is None:
        return ""
    return str(val).strip().replace("\\", "\\\\").replace("'", "\\'")

def run_sql(label: str, stmt: str) -> bool:
    try:
        spark.sql(stmt)
        log(f"    [SUCCESS] {label}")
        return True
    except Exception as err:
        log(f"    [WARNING] {label} failed: {str(err).splitlines()[0]}")
        return False

def append_log_event(dataset_id: str, title: str, category: str, portal_update: str,
                     status: str, records: int = 0, error: str = None):
    """
    Appends an immutable audit row to the queue table instead of updating in place.
    Preserves audit history and prevents write lock contention.
    """
    schema = StructType([
        StructField("dataset_id", StringType(), False),
        StructField("title", StringType(), True),
        StructField("category", StringType(), True),
        StructField("last_portal_update", StringType(), True),
        StructField("status", StringType(), True),
        StructField("records_ingested", LongType(), True),
        StructField("error_message", StringType(), True)
    ])

    log_row = [{
        "dataset_id": dataset_id,
        "title": title,
        "category": category,
        "last_portal_update": portal_update,
        "status": status,
        "records_ingested": records,
        "error_message": str(error)[:1000] if error else None
    }]

    (spark.createDataFrame(log_row, schema=schema)
        .withColumn("logged_at", current_timestamp())
        .select(
            "dataset_id", "title", "category", "last_portal_update",
            "status", "records_ingested", "error_message", "logged_at"
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(QUEUE_TABLE))

In [0]:
# COMMAND ----------
# CELL 2: Ingestion Logic for Single Dataset with Append-Only Logging
SECRET_SCOPE = "socrata_scope"
try:
    APP_TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key="app_token")
    log("Loaded Socrata app token from secret scope.")
except Exception:
    APP_TOKEN = None
    log("Warning: Running unauthenticated.")

client = Socrata(SOCRATA_DOMAIN, APP_TOKEN, timeout=600)

MAX_TAGS = 50
GEO_TYPES = {
    'point', 'multipoint', 'linestring', 'multilinestring', 
    'polygon', 'multipolygon', 'location', 'geometry'
}

def parse_geo_value(val):
    if val is None:
        return None
    if isinstance(val, (dict, list)):
        return json.dumps(val)
    val_str = str(val).strip()
    return val_str if val_str else None

log(f"\nProcessing Single Dataset Task: {DATASET_ID}")

# 1. Fetch Metadata
try:
    dataset_metadata = client.get_metadata(DATASET_ID)
except Exception as err:
    err_msg = str(err).splitlines()[0]
    append_log_event(
        dataset_id=DATASET_ID,
        title=DATASET_ID,
        category="Uncategorized",
        portal_update=None,
        status="FAILED",
        records=0,
        error=f"Metadata fetch failed: {err_msg}"
    )
    raise RuntimeError(f"Metadata fetch failed for {DATASET_ID}: {err}")

dataset_title = dataset_metadata.get('name', DATASET_ID)
category = dataset_metadata.get("category") or "Uncategorized"
portal_updated_at = (dataset_metadata.get("rowsUpdatedAt") or 
                     dataset_metadata.get("dataUpdatedAt"))

# 2. Audience / Publication check
approvals = (dataset_metadata.get("approvals") or [{}])[0] or {}
target_audience = (approvals.get("targetAudience") or "").lower()
publication_stage = (dataset_metadata.get("publicationStage") or "").lower()

if not (publication_stage == "published" and target_audience == "public"):
    reason = f"Filtered out: publicationStage={publication_stage}, targetAudience={target_audience}"
    log(reason)
    append_log_event(
        dataset_id=DATASET_ID,
        title=dataset_title,
        category=category,
        portal_update=portal_updated_at,
        status="SKIPPED",
        records=0,
        error=reason
    )
    dbutils.notebook.exit("Filtered out non-public dataset.")

table_name = sanitize_table_name(dataset_title)
full_table_path = f"`{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{table_name}`"
log(f"Target Delta Table: {full_table_path}")

socrata_columns = dataset_metadata.get('columns', [])
all_meta_columns = [c.get('fieldName') for c in socrata_columns if c.get('fieldName')]

geo_cols = set()
socrata_type_cast_map = {}

for c in socrata_columns:
    fn = c.get('fieldName')
    dt = (c.get('dataTypeName') or '').lower()
    if not fn:
        continue
    if dt in GEO_TYPES:
        geo_cols.add(fn)
    elif dt in ['number', 'money']:
        socrata_type_cast_map[fn] = 'double'
    elif dt == 'checkbox':
        socrata_type_cast_map[fn] = 'boolean'

serverless_input_schema = StructType([
    StructField(col_name, StringType(), True) for col_name in all_meta_columns
])
cast_exprs = {c: col(c).cast(t) for c, t in socrata_type_cast_map.items()}

# 3. Dynamic Batch Sizing and Data Extraction Loop
offset = 0
first_batch = True
total_records = 0

col_count = len(all_meta_columns)
if col_count > 50 or geo_cols:
    BATCH_SIZE = 10000
elif col_count > 25:
    BATCH_SIZE = 25000
else:
    BATCH_SIZE = 50000

try:
    while True:
        try:
            records = client.get(DATASET_ID, limit=BATCH_SIZE, offset=offset, order=":id")
        except Exception:
            records = client.get(DATASET_ID, limit=BATCH_SIZE, offset=offset)

        if not records:
            break

        cleaned_rows = []
        for r in records:
            row_dict = {}
            for col_name in all_meta_columns:
                val = r.get(col_name)
                if val is None:
                    row_dict[col_name] = None
                elif col_name in geo_cols:
                    row_dict[col_name] = parse_geo_value(val)
                elif isinstance(val, (dict, list)):
                    row_dict[col_name] = json.dumps(val)
                else:
                    val_str = str(val).strip()
                    if col_name in socrata_type_cast_map and socrata_type_cast_map[col_name] == 'double':
                        val_str = re.sub(r'[\$,]', '', val_str)
                        row_dict[col_name] = val_str if val_str else None
                    else:
                        row_dict[col_name] = val_str if val_str else None
            cleaned_rows.append(row_dict)

        chunk_df = spark.createDataFrame(cleaned_rows, schema=serverless_input_schema)
        if cast_exprs:
            chunk_df = chunk_df.withColumns(cast_exprs)

        if first_batch:
            chunk_df.write \
                .format("delta") \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .saveAsTable(full_table_path)
        else:
            chunk_df.write \
                .format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(full_table_path)

        total_records += len(records)
        offset += BATCH_SIZE
        first_batch = False

        if len(records) < BATCH_SIZE:
            break

    log(f"Committed {total_records} rows into {full_table_path}.")

    # 4. OPTIMIZE & Metadata
    if total_records > 0:
        try:
            spark.sql(f"OPTIMIZE {full_table_path}")
        except Exception as opt_err:
            log(f"Warning: OPTIMIZE failed: {opt_err}")

        md = dataset_metadata.get("metadata") or {}
        custom_fields = md.get("custom_fields") or {}
        time_period_custom = custom_fields.get("Time Period") or {}
        agency_custom = custom_fields.get("Agency") or {}
        jurisdiction_custom = custom_fields.get("Jurisdiction") or {}
        gis_custom = custom_fields.get("GIS Download") or {}
        data_provided_by = dataset_metadata.get("attribution") or ""

        run_sql("Table Comment", f"COMMENT ON TABLE {full_table_path} IS '{sql_str((dataset_metadata.get('description') or ''))}'")

        tbl_properties = {
            "open_data.title": dataset_metadata.get("name"),
            "open_data.url": f"https://{SOCRATA_DOMAIN}/d/{DATASET_ID}",
            "open_data.category": category,
            "open_data.agency": agency_custom.get("State Agency Performing Data Updates", data_provided_by),
            "open_data.jurisdiction": jurisdiction_custom.get("Jurisdiction", "State of Maryland"),
            "open_data.update_frequency": time_period_custom.get("Update Frequency"),
            "open_data.time_period_of_content": time_period_custom.get("Time Period of Content"),
            "open_data.date_metadata_written": time_period_custom.get("Date Metadata Written"),
            "open_data.data_provided_by": data_provided_by,
            "open_data.source_link": dataset_metadata.get("attributionLink"),
            "open_data.row_label": md.get("rowLabel", "Row"),
            "open_data.place_keywords": gis_custom.get("Place Keywords", "Maryland"),
            "open_data.public_license_type": (dataset_metadata.get("license") or {}).get("name", "Public Domain"),
            "open_data.contact_owner": (dataset_metadata.get("owner") or {}).get("displayName"),
            "open_data.provenance": (dataset_metadata.get("provenance") or "").lower(),
            "open_data.publicationStage": publication_stage,
            "open_data.targetAudience": target_audience,
            "socrata.domain": SOCRATA_DOMAIN,
            "socrata.dataset_id": DATASET_ID,
            "source.system": "Socrata Open Data Portal",
        }
        prop_clauses = [f"'{sql_str(k)}' = '{sql_str(v)}'" for k, v in tbl_properties.items() if sql_str(v)]
        if prop_clauses:
            props_sql = ",\n        ".join(prop_clauses)
            run_sql(f"{len(prop_clauses)} Properties", f"ALTER TABLE {full_table_path} SET TBLPROPERTIES (\n        {props_sql}\n)")

        raw_tags = dataset_metadata.get("tags") or []
        tag_candidates = [re.sub(r'[^a-zA-Z0-9_]', '_', str(t).strip().lower()).strip('_') for t in list(raw_tags) + [category]]
        unique_tags = list(dict.fromkeys([t for t in tag_candidates if t]))[:MAX_TAGS]
        if unique_tags:
            formatted_tags = ", ".join(f"'{sql_str(tag)}'" for tag in unique_tags)
            run_sql(f"{len(unique_tags)} Tags", f"ALTER TABLE {full_table_path} SET TAGS ({formatted_tags})")

        for col_meta in socrata_columns:
            fn = col_meta.get("fieldName")
            if fn in all_meta_columns:
                cmt = col_meta.get("description") or col_meta.get("name") or fn
                try:
                    spark.sql(f"ALTER TABLE {full_table_path} ALTER COLUMN `{fn}` COMMENT '{sql_str(str(cmt)[:1000])}'")
                except Exception:
                    pass

    # 5. Append SUCCESS to Audit Log
    append_log_event(
        dataset_id=DATASET_ID,
        title=dataset_title,
        category=category,
        portal_update=portal_updated_at,
        status="SUCCESS",
        records=total_records,
        error=None
    )
    log(f"Successfully appended SUCCESS audit log entry for {DATASET_ID}.")

except Exception as pipeline_err:
    err_msg = str(pipeline_err).splitlines()[0]
    append_log_event(
        dataset_id=DATASET_ID,
        title=dataset_title,
        category=category,
        portal_update=portal_updated_at,
        status="FAILED",
        records=0,
        error=err_msg
    )
    raise RuntimeError(f"Ingestion failed for {DATASET_ID}: {pipeline_err}")